# 01a0 — Blockchain & smart contracts primer

A ground-up, hands-on tour. We drive a real local Ethereum chain (`anvil`) through `cast` and `forge`, naming each concept after we've executed it. By the last section you'll be able to read [BandwidthEscrow.sol](../../contracts/src/BandwidthEscrow.sol) line by line.

**Prereq:** `anvil`, `cast`, `forge` on PATH (Foundry installed). Run cells top to bottom — anvil is started in the next cell and killed in the very last cell.

In [1]:
# --- Notebook runtime setup ---------------------------------------
import atexit, subprocess, time, shutil, sys, pathlib, json, os

PRIMER_DIR = pathlib.Path.cwd().resolve()
REPO_ROOT = PRIMER_DIR.parent.parent
RPC = 'http://127.0.0.1:8545'

def run(cmd, cwd=None, check=True):
    """Run a shell command, show it, return stdout."""
    print('$', ' '.join(str(c) for c in cmd))
    r = subprocess.run(cmd, cwd=cwd or PRIMER_DIR, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.returncode != 0:
        if r.stderr: print(r.stderr.rstrip(), file=sys.stderr)
        if check: raise SystemExit(f'command failed: {cmd}')
    return r.stdout.strip()

for tool in ('anvil', 'cast', 'forge'):
    assert shutil.which(tool), f'{tool} not found on PATH'
print('Foundry tools OK')

Foundry tools OK


In [2]:
# --- Start anvil --------------------------------------------------
_anvil_proc = subprocess.Popen(
    ['anvil', '--host', '127.0.0.1', '--port', '8545', '--silent'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(_anvil_proc.terminate)

# Wait for RPC to respond.
for _ in range(30):
    try:
        run(['cast', 'block-number', '--rpc-url', RPC], check=True)
        break
    except SystemExit:
        time.sleep(0.2)
else:
    raise RuntimeError('anvil did not come up')
print(f'anvil PID={_anvil_proc.pid}')

$ cast block-number --rpc-url http://127.0.0.1:8545


Error: error sending request for url (http://127.0.0.1:8545/)

Context:
- Error #0: client error (Connect)
- Error #1: tcp connect error
- Error #2: Connection refused (os error 111)


$ cast block-number --rpc-url http://127.0.0.1:8545
0
anvil PID=139381


## Teardown

Kill the anvil process. Re-run this notebook from the top to start fresh.

In [3]:
_anvil_proc.terminate()
_anvil_proc.wait(timeout=5)
print('anvil stopped')

anvil stopped
